In [ ]:
### conda activate mrvi-env

import scanpy as sc
import scvi
from scvi.external import MRVI
scvi.settings.seed = 42

import pandas as pd

# define functions

In [ ]:
def add_genotype_column(adata, variant, min_counts):
    if variant + '_mutated' in adata.obsm['genotypes'].columns:
        genotype_cols = [variant + '_wt', variant + '_mutated', variant + '_heterozygous']
        genotype_data = adata.obsm['genotypes'][genotype_cols]
        
        # Handle all-NA rows to avoid FutureWarning
        all_na_mask = genotype_data.isna().all(axis=1)
        genotypes = pd.Series(index=genotype_data.index, dtype='object')
        
        # Only call idxmax on rows that have at least one non-NA value
        if not all_na_mask.all():
            genotypes[~all_na_mask] = genotype_data[~all_na_mask].idxmax(axis=1)
        
        # Clean up the column names
        genotypes = genotypes.str.replace(variant + '_', '', regex=False)
        adata.obs['genotype'] = genotypes
        
    elif variant + '_wt' in adata.obsm['genotypes'].columns:
        genotype_data = adata.obsm['genotypes'][[variant + '_wt']]
        
        # Handle all-NA rows
        all_na_mask = genotype_data.isna().all(axis=1)
        genotypes = pd.Series(index=genotype_data.index, dtype='object')
        
        if not all_na_mask.all():
            genotypes[~all_na_mask] = genotype_data[~all_na_mask].idxmax(axis=1)
        
        genotypes = genotypes.str.replace(variant + '_', '', regex=False)
        adata.obs['genotype'] = genotypes
    
    # Apply the min_counts filter
    adata.obs.loc[adata.obsm['genotypes'][variant + '_high_confidence_counts'] < min_counts, 'genotype'] = None
    return adata

# Open and filter adatas
h5ad objects not provided but can be recreated from cellranger outputs (GEO) using:
- 00_example_cellranger_to_filtered_clustered.ipynb - example to go from cellranger output to processed (filtered, clustered, etc) h5ad 
- ../../3_figure_FFPE/code/00_probabilistic_genotype_assignment - example to go from processed h5ad to genotype assignments

In [ ]:
dir = '' ### set to directory where data is stored

adatas = []
kept_samples = []

variant = 'JAK2 c.1849G>T'

for lib in ['1-plex','4-plex','1','2']:
    if lib == '4-plex':
        BCs = ['BC001', 'BC002', 'BC003', 'BC004']
    elif lib == '1-plex':
        BCs = ['BC001']
    else:
        BCs = ['BC001', 'BC002', 'BC003', 'BC004', 'BC005', 'BC006', 'BC007', 'BC008', 'BC009', 'BC010', 'BC011', 'BC012', 'BC013', 'BC014', 'BC015', 'BC016']
    for BC in BCs:
        if lib == '4-plex':
            adata_dir = dir + 'MPN_4plex_' + BC + '_genotyped.h5ad'
        elif lib == '1-plex':
            adata_dir = dir + 'MPN_1plex_' + BC + '_genotyped.h5ad'
        else:
            adata_dir = dir + 'MPN_' + lib + '_' + BC + '_genotyped.h5ad'
        adata = sc.read_h5ad(adata_dir)

        keep_all = True
        adata = add_genotype_column(adata, variant, 2) ### require min counts of 2
        genotype_proportions = adata[~adata.obs['cell_type'].isin(['T cell (non-HSPC)','B cell (non-HSPC)'])].obs['genotype'].value_counts(normalize=True)
        if ('heterozygous' in genotype_proportions.index) and ('wt' in genotype_proportions.index) and ('mutated' in genotype_proportions.index):
            if (genotype_proportions['wt'] > 0.05) & (genotype_proportions['mutated'] > 0.05):
                adata_filtered = adata[(adata.obs['genotype'].notna())].copy()
                del adata
                del adata_filtered.obsm['genotypes']
                del adata_filtered.obsm['X_pca']
                del adata_filtered.obsm['X_umap']
                adatas.append(adata_filtered)
                kept_samples.append(BC + '_' + lib)
                print(BC + '_' + lib)

adata = sc.concat(adatas, label='sample', keys=kept_samples, index_unique='-', join='inner')
del adatas

adata.obs['sample_genotype'] = adata.obs['sample'].astype(str) + '_' + adata.obs['genotype'].astype(str)

### remove samples with fewer than 100 cells
sample_counts = adata.obs['sample'].value_counts()
samples_included = sample_counts[sample_counts > 100].index.tolist()
adata = adata[adata.obs['sample'].isin(samples_included)].copy()

genotype_counts = adata.obs.groupby('sample')['genotype'].value_counts().unstack().fillna(0).astype(int)
samples_included = adata.obs['sample'].unique().tolist()

In [ ]:
## clean up cell type annotations
adata = adata[adata.obs['cell_type'] != 'Monocyte/Platelet doublet (non-HSPC)'].copy()
adata.obs.loc[adata.obs['cell_type'] == 'Classical monocyte 2 (non-HSPC)', 'cell_type'] = 'Classical monocyte (non-HSPC)'

# run MrVI

In [ ]:
adata.X = adata.layers['raw_data']

sc.pp.highly_variable_genes(
    adata, n_top_genes=3000, inplace=True, subset=False, flavor="seurat_v3", batch_key='sample'
)

adata = adata[:,(adata.var['highly_variable_nbatches'] >= 5)].copy()

MRVI.setup_anndata(adata, sample_key='genotype', batch_key='sample')
model = MRVI(adata)
model.train(max_epochs=300, early_stopping=True)


In [ ]:
u = model.get_latent_representation()
adata.obsm["u"] = u
sc.pp.neighbors(adata, use_rep="u", key_added="neighbors_u")
sc.tl.umap(adata, min_dist=0.3, neighbors_key="neighbors_u", key_added="X_umap_u")

sc.pl.embedding(
    adata,
    basis="X_umap_u",
    color=["genotype"],
    frameon=False,
    ncols=1
)

v = model.get_latent_representation(give_z=True)
adata.obsm["v"] = v
sc.pp.neighbors(adata, use_rep="v", key_added="neighbors_v")
sc.tl.umap(adata, min_dist=0.3, neighbors_key="neighbors_v", key_added="X_umap_v")

sc.pl.embedding(
    adata,
    basis="X_umap_v",
    color=["genotype"],
    frameon=False,
    ncols=1
)


In [ ]:
### save h5ad as mrvi_jak2_heterozygous_included.h5ad (provided)
